# Example 1

## Description

A Markov Chain is composed of
+ a **state space**
+ a **transition structure** (probability matrix or infinitesimal generator)
+ an **initial distribution** of the state

In this first lesson, we show various ways of creating such objects. We will in particular highlight *transition structures* and the class `MarkovChain`.

This example creates a 3-state, discrete-time Markov chain, then performs a (Monte-Carlo) simulation of it. Usage:

Here, n is the number of steps for the simulation, and p1, p2, p3 are the respective initial probabilities of the three states.

## Tasks performed

* create a `DiscreteDistribution` object to hold the initial distribution of the process

* create a `SparseMatrix` object to hold the transition matrix of the chain, entry by entry with the `addToEntry()` function;

* create a `MarkovChain` object and link the previous elements to it;

* output the Markov chain object to the screen;

* create a simulation of a trajectory and store it in a `SimulationResult` object;

* write the trajectory to the screen;

* clean up.

In [1]:
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMarkovChain")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")

// Charger explicitement les libs .so
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")


## First example: a discrete-time Markov chain with 3 states

We make sure we include the marmote libraries needed for that example : 

In [2]:
#include <marmoteMarkovChain/marmoteMarkovChain.h>
#include <marmoteMarkovChain/marmoteSimulationResult.h>
#include <marmoteCore/marmoteDiscreteDistribution.h>
#include <marmoteCore/marmoteSparseMatrix.h>
#include <iostream>

This is a forward declaration of utility, if you want to give parameters to your program.

### State space

We fix the number of iterations here, and create variables with random values.

In [3]:
unsigned int n = 10;
double prob1 = 0.1, prob2 = 0.2, prob3 = 0.7;

You can use alternatively the dedicated MarmoteInterval structure of MarmoteCore.
We create here the initial distribution vector of size 3. Then, we fill the array with the previous variables. 

In [4]:
double states[3] = {0, 1, 2};
double* probas = new double[3]{prob1, prob2, prob3};

### Transition structure

Now, we create a discrete distribution for the three states and then, a discrete-time Markov chain of size 3. 

In [5]:
DiscreteDistribution* initial = new DiscreteDistribution(3, states, probas);
MarkovChain* c1 = new MarkovChain(3, DISCRETE);

We assign here the created distribution to the chain.

In [6]:
c1->set_init_distribution(initial);

We create the transition matrix with size 3*3 using `sparseMatrix` object. We also define the type of the transition structure. By default, the type for a sparse matrix is not defined.

In [7]:
SparseMatrix* P = new SparseMatrix(3);

`Marmote` has to know if this is a discrete-time or continuous-time transition structure.

In [8]:
P->set_type(DISCRETE);

Now fill the values in. Two method are available for this: `setEntry` and `addToEntry`.
Both have parameters `(row,column,value)`.

In [9]:
P->addToEntry(0,0,0.25); P->addToEntry(0,1,0.5);  P->addToEntry(0,2,0.25);
P->addToEntry(1,0,0.4);  P->addToEntry(1,1,0.2);  P->addToEntry(1,2,0.4);
P->addToEntry(2,0,0.4);  P->addToEntry(2,1,0.3);  P->addToEntry(2,2,0.3);

Now, we assign the transition matrix to the chain

In [10]:
c1->set_generator(P);

Now, let's print chain information to the terminal stdout.

In [11]:
c1->Write(&std::cout);

discrete sparse
3
         0          0 2.500000e-01
         0          1 5.000000e-01
         0          2 2.500000e-01
         1          0 4.000000e-01
         1          1 2.000000e-01
         1          2 4.000000e-01
         2          0 4.000000e-01
         2          1 3.000000e-01
         2          2 3.000000e-01
stop
discrete values { 0 1 2 } probas {      0.1      0.2      0.7 } 


We can create a simulationResult object to save results of simulated chain.

In [12]:
SimulationResult* simRes1 = c1->SimulateChainDT(n, false, true, false);

Write the trajectory of the marmote in the terminal stdout

In [13]:
std::cout << "Trajectory\n";
simRes1->WriteTrajectory(&std::cout, "standard");
std::cout << std::endl;

Trajectory
         0        1 1
         1        2 2
         2        2 2
         3        1 1
         4        0 0
         5        0 0
         6        1 1
         7        0 0
         8        1 1
         9        2 2
        10        2 2



Now, we can destroy the simulation object, the chain and what it remains in order to free the allocated memory.

In [14]:
delete simRes1;
delete c1;
delete initial;
delete[] probas;

## Output

The output consists in:
* the code of the Markov chain, which itself consists in:
  * the description of the matrix, here an entry-by-entry format
  * the description of the initial distribution

* the trajectory, where each line represents a time instant: three numbers appear: 1) the time step; 2) the state index; 3) the state description (in this case, the state and the index are the same).

